In [ ]:
import torch
from torchvision import datasets, transforms

# 1. Define transformations
# Convert images to tensors and normalize pixel values to range [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 2. Download and load the training data
train_dataset = datasets.MNIST(root='./data', 
                               train=True, 
                               download=True, 
                               transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, 
                                           batch_size=64, 
                                           shuffle=True)

# 3. Download and load the test data
test_dataset = datasets.MNIST(root='./data', 
                              train=False, 
                              download=True, 
                              transform=transform)

test_loader = torch.utils.data.DataLoader(test_dataset, 
                                          batch_size=64, 
                                          shuffle=False)

print(f"Downloaded {len(train_dataset)} training images and {len(test_dataset)} testing images.")

In [ ]:
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        
        # ENCODER: Compresses the 28x28 image (784 pixels) into a smaller latent vector
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 12)  # 12 is the size of our compressed latent space
        )
        
        # DECODER: Reconstructs the 784-pixel image from the latent vector
        self.decoder = nn.Sequential(
            nn.Linear(12, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Tanh() # We use Tanh because our input normalization scaled values between [-1, 1]
        )

    def forward(self, x):
        # 1. Flatten the image from (batch_size, 1, 28, 28) to (batch_size, 784)
        x = x.view(-1, 28 * 28)
        
        # 2. Pass through encoder
        encoded = self.encoder(x)
        
        # 3. Pass through decoder
        decoded = self.decoder(encoded)
        
        # 4. Reshape back to an image format (batch_size, 1, 28, 28)
        decoded = decoded.view(-1, 1, 28, 28)
        
        return decoded

# Instantiate the model
model = Autoencoder()
print(model)

In [ ]:
import matplotlib.pyplot as plt

def plot_mnist_batch(dataloader, num_images=5):
    """Fetches a batch from the dataloader and plots the first few images."""
    # Get one batch of images
    dataiter = iter(dataloader)
    images, labels = next(dataiter)
    
    # We normalized the images to [-1, 1]. To display them correctly, 
    # we need to un-normalize them back to [0, 1]
    images = images / 2 + 0.5 
    
    # Set up the plot
    fig, axes = plt.subplots(1, num_images, figsize=(12, 3))
    for i in range(num_images):
        # Squeeze removes the channel dimension (1, 28, 28) -> (28, 28) for plotting
        axes[i].imshow(images[i].numpy().squeeze(), cmap='gray')
        axes[i].set_title(f"Label: {labels[i].item()}")
        axes[i].axis('off')
        
    plt.show()

# Example usage (assuming train_loader is defined):
# plot_mnist_batch(train_loader)

In [ ]:
from PIL import Image
from torchvision import transforms

def predict_from_filepath(filepath, model):
    """Loads an image from a file, passes it through the autoencoder, and plots both."""
    
    # 1. Load the image and convert to Grayscale ('L')
    try:
        img = Image.open(filepath).convert('L')
    except FileNotFoundError:
        print(f"Error: Could not find image at '{filepath}'")
        return

    # 2. Define the exact same transforms used during training
    transform = transforms.Compose([
        transforms.Resize((28, 28)), # Force it to be 28x28 pixels
        transforms.ToTensor(),       # Convert to PyTorch Tensor
        transforms.Normalize((0.5,), (0.5,)) # Normalize to [-1, 1]
    ])
    
    # 3. Apply transforms and add a batch dimension (B, C, H, W)
    # Becomes shape: (1, 1, 28, 28)
    input_tensor = transform(img).unsqueeze(0) 
    
    # 4. Pass through the model to get the reconstruction
    model.eval() # Set model to evaluation mode (important!)
    with torch.no_grad():
        reconstructed_tensor = model(input_tensor)
        
    # 5. Un-normalize the tensors back to [0, 1] for plotting
    original_img_plot = input_tensor.squeeze().numpy() / 2 + 0.5
    reconstructed_img_plot = reconstructed_tensor.squeeze().numpy() / 2 + 0.5
    
    # 6. Plot Original vs. Reconstructed side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    
    axes[0].imshow(original_img_plot, cmap='gray')
    axes[0].set_title("Original Input")
    axes[0].axis('off')
    
    axes[1].imshow(reconstructed_img_plot, cmap='gray')
    axes[1].set_title("Model Reconstruction")
    axes[1].axis('off')
    
    plt.show()

# Example usage:
# my_model = Autoencoder()
# # ... load your trained weights here if you have them ...
# predict_from_filepath("my_handwritten_digit.png", my_model)

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset

def load_image(img_path):
    # Convert to RGB to ensure consistency before applying transforms
    img = Image.open(img_path).convert('RGB')

    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1), # Forces 1 channel (grayscale)
        transforms.Resize((28, 28)),                 # Safely resizes to 28x28
        transforms.ToTensor()                        # Converts to Tensor and scales to [0.0, 1.0]
    ])
    
    img_tensor = transform(img)
    return img_tensor

def load_images():
    train_data = []
    images = ["bright.png", "kid.png", "tiger.png"]
    
    for img_name in images:
        img_path = os.path.join(os.getcwd(), "data", "exam", img_name)
        
        # Added a safety check to ensure the file actually exists
        if os.path.exists(img_path):
            img_tensor = load_image(img_path)
            train_data.append(img_tensor)
        else:
            print(f"Warning: File not found -> {img_path}")
            
    # Use torch.stack instead of np.vstack since we are dealing with Tensors
    if train_data:
        train_data = torch.stack(train_data)
    else:
        train_data = torch.empty(0)
        
    return train_data

# Notice: I removed `reshape_images()` entirely because `transforms` handles 
# the resizing, grayscaling, and scaling much more cleanly and safely.

train_data = load_images()
if train_data.numel() > 0:
    print(f"Train data shape: {train_data.shape}") # Should print: torch.Size([3, 1, 28, 28])

class DenoisingImages(Dataset):
    def __init__(self, data, noise_factor=0.1):
        # Safely handle data whether it's passed as a Tensor or Numpy array
        if isinstance(data, torch.Tensor):
            self.data = data.clone().detach()
        else:
            self.data = torch.tensor(data, dtype=torch.float32)
            
        self.noise_factor = noise_factor
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        clean = self.data[index]
        noise = self.noise_factor * torch.randn_like(clean)
        
        # Since ToTensor() scales to [0, 1], clamping between 0 and 1 is perfect
        noisy = torch.clamp(clean + noise, 0.0, 1.0) 
        
        return noisy, clean

In [ ]:
train_losses = []
for epoch in range(1):
    running_loss = 0.0
    model.train()
    
    for noisy, clean in trainloader:
        noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(noisy)

        loss = criterion(outputs, clean)
        
        loss.backward()
        
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(trainloader)
    train_losses.append(epoch_loss)
    
    print(f"Epoch [{epoch + 1}] Train Loss: {epoch_loss:.4f}")